In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

In [ ]:
def simulate_qcare(alpha, mu1, mu2, T):
    """
    Simulating an agent playing QCARE for T rounds.
    
    alpha: exploration parameter
    mu1, mu2: true reward probabilities of arm 1 and arm 2
    T: number of rounds
    
    Returns total regret
    """
    # Track pulls and empirical means for each arm
    k = np.zeros(2)      # number of pulls
    mu_hat = np.zeros(2) # empirical means
    
    best_mu = max(mu1, mu2)  # optimal expected reward
    regret = 0
    
    for t in range(T):
        # Generate scores for each arm
        noise = np.random.normal(0, 1, 2)
        scores = mu_hat + noise / (k + 1)**alpha
        
        # Choose arm with highest score
        chosen = np.argmax(scores)
        
        # Observe binary reward
        true_mu = mu1 if chosen == 0 else mu2
        reward = np.random.binomial(1, true_mu)
        
        # Update regret
        regret += best_mu - true_mu
        
        # Update statistics
        k[chosen] += 1
        mu_hat[chosen] = ((mu_hat[chosen] * (k[chosen] - 1)) + reward) / k[chosen]
    
    return regret

In [4]:
def average_regret(alpha, mu1, mu2, T=200, n_sims=10000):
    """
    Average regret over many simulations for a given alpha and reward gap.
    """
    regrets = [simulate_qcare(alpha, mu1, mu2, T) for _ in range(n_sims)]
    return np.mean(regrets)


In [7]:
# Defining the reward gap configurations
reward_gaps = [
    (0.45, 0.55),  # Smallest gap
    (0.4, 0.6),    # Small gap 
    (0.3, 0.7),    # Fairly large gap
    (0.1, 0.9)     # Largest gap
]

# Define alpha values to test
alphas = np.arange(0.1, 2.1, 0.1)

results = {}

for (mu1, mu2) in reward_gaps:
    label = f"({mu1}, {mu2})"
    print(f"Running simulations for {label}...")
    results[label] = []
    for alpha in alphas:
        avg_reg = average_regret(alpha, mu1, mu2, T=200, n_sims=5000)
        results[label].append(avg_reg)
    print(f"Done with {label}")

print("Done")

Running simulations for (0.45, 0.55)...
Done with (0.45, 0.55)
Running simulations for (0.4, 0.6)...
Done with (0.4, 0.6)
Running simulations for (0.3, 0.7)...
Done with (0.3, 0.7)
Running simulations for (0.1, 0.9)...
Done with (0.1, 0.9)
Done
